In [ ]:
import os
import json
import time
from math import radians, sin, cos, asin, sqrt
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from tqdm import tqdm

# ============================================================
# 0. KONFIGURASI GLOBAL (VERSI MAC)
# ============================================================

BMKG_PATH = "/Volumes/Extreme SSD/Indonesian Earthquake Catalog (BMKG), 1998–2024/BMKG_Earthquake_Catalog.csv"
USGS_PATH = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/usgs_katalog/katalog_usgs_master_2001_2025.csv"
RADAR_JSON_PATH = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/radar_inventory_3ch_detailed.json"

SAVE_DIR = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/waveforms_hybrid"
LOG_PATH = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/hybrid_download_log.csv"
DASHBOARD_HTML = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/hybrid_dashboard.html"

os.makedirs(SAVE_DIR, exist_ok=True)

client_iris = Client("EARTHSCOPE", timeout=30)
client_gfz = Client("GFZ", timeout=30)

TIME_TOL = pd.Timedelta("120s")
DIST_TOL_KM = 50
RADAR_MATCH_TOL = pd.Timedelta("2s")
MAX_WORKERS = 16  # multi-threaded downloader


# ============================================================
# 1. UTILITAS
# ============================================================

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c


# ============================================================
# 2. LOAD & NORMALISASI KATALOG BMKG & USGS (FIX TZ)
# ============================================================

def load_catalogs():
    print("📥 Load katalog BMKG & USGS...")

    bmkg = pd.read_csv(BMKG_PATH)
    bmkg["origin_time"] = pd.to_datetime(
        bmkg["Date"] + " " + bmkg["Time (UTC)"],
        errors="coerce",
        utc=True
    )

    usgs = pd.read_csv(USGS_PATH)
    usgs["time"] = pd.to_datetime(
        usgs["time"],
        format="mixed",
        errors="coerce",
        utc=True
    )

    bmkg = bmkg[
        bmkg["Longitude"].between(92, 142) &
        bmkg["Latitude"].between(-11, 14)
    ].copy()

    usgs = usgs[
        usgs["longitude"].between(92, 142) &
        usgs["latitude"].between(-11, 14)
    ].copy()

    print(f"   BMKG events: {len(bmkg)}")
    print(f"   USGS events: {len(usgs)}")
    return bmkg, usgs


# ============================================================
# 3. BUILD KATALOG HYBRID BMKG ↔ USGS
# ============================================================

def build_hybrid_catalog(bmkg, usgs):
    print("🔗 Matching BMKG → USGS...")

    matches = []

    for idx, row in tqdm(bmkg.iterrows(), total=len(bmkg), desc="Matching", unit="ev"):
        t_b = row["origin_time"]
        lat_b = row["Latitude"]
        lon_b = row["Longitude"]

        cand = usgs[
            (usgs["time"] >= t_b - TIME_TOL) &
            (usgs["time"] <= t_b + TIME_TOL)
        ].copy()

        if cand.empty:
            matches.append({"bmkg_index": idx, "usgs_index": None})
            continue

        cand["dist_km"] = cand.apply(
            lambda r: haversine(lat_b, lon_b, r["latitude"], r["longitude"]),
            axis=1
        )
        cand = cand[cand["dist_km"] <= DIST_TOL_KM]

        if cand.empty:
            matches.append({"bmkg_index": idx, "usgs_index": None})
            continue

        cand["dt_abs"] = (cand["time"] - t_b).abs()
        best = cand.sort_values(["dist_km", "dt_abs"]).iloc[0]

        matches.append({
            "bmkg_index": idx,
            "usgs_index": best.name
        })

    hybrid_rows = []
    for m in matches:
        b = bmkg.loc[m["bmkg_index"]]

        if m["usgs_index"] is None:
            hybrid_rows.append({
                "source": "BMKG_ONLY",
                "bmkg_index": int(m["bmkg_index"]),
                "usgs_index": None,
                "bmkg_time": b["origin_time"],
                "bmkg_lat": b["Latitude"],
                "bmkg_lon": b["Longitude"],
                "bmkg_mag": b["Magnitude"],
                "usgs_time": None,
                "usgs_lat": None,
                "usgs_lon": None,
                "usgs_mag": None,
                "usgs_depth": None
            })
        else:
            u = usgs.loc[m["usgs_index"]]
            hybrid_rows.append({
                "source": "BMKG+USGS",
                "bmkg_index": int(m["bmkg_index"]),
                "usgs_index": int(m["usgs_index"]),
                "bmkg_time": b["origin_time"],
                "bmkg_lat": b["Latitude"],
                "bmkg_lon": b["Longitude"],
                "bmkg_mag": b["Magnitude"],
                "usgs_time": u["time"],
                "usgs_lat": u["latitude"],
                "usgs_lon": u["longitude"],
                "usgs_mag": u["mag"],
                "usgs_depth": u["depth"]
            })

    hybrid = pd.DataFrame(hybrid_rows)
    print(f"   Hybrid total: {len(hybrid)}")
    print(f"   BMKG+USGS   : {(hybrid['source']=='BMKG+USGS').sum()}")
    print(f"   BMKG_ONLY   : {(hybrid['source']=='BMKG_ONLY').sum()}")
    return hybrid


# ============================================================
# 4. LOAD RADAR JSON
# ============================================================

def load_radar_events():
    with open(RADAR_JSON_PATH, "r") as f:
        events = json.load(f)
    print(f"📡 Radar events: {len(events)}")
    return events


# ============================================================
# 5. LOOKUP HYBRID
# ============================================================

def build_hybrid_lookup(hybrid):
    return {int(row["bmkg_index"]): row for _, row in hybrid.iterrows()}


# ============================================================
# 6. STATION HEALTH CACHE (AUTO-DETECT STASIUN AKTIF)
# ============================================================

station_health_cache = {}  # sta_code -> "ACTIVE" / "NO_DATA" / "UNKNOWN"

def check_station_active(sta_code):
    if sta_code in station_health_cache:
        return station_health_cache[sta_code]

    # Cek dengan window dummy kecil di masa lalu
    try:
        t_dummy = UTCDateTime("2015-01-01T00:00:00")
        for client in (client_iris, client_gfz):
            try:
                st = client.get_waveforms(
                    network="*",
                    station=sta_code,
                    location="*",
                    channel="BH?,HH?,EH?",
                    starttime=t_dummy,
                    endtime=t_dummy + 60
                )
                if len(st) > 0:
                    station_health_cache[sta_code] = "ACTIVE"
                    return "ACTIVE"
            except:
                continue
    except:
        pass

    station_health_cache[sta_code] = "NO_DATA"
    return "NO_DATA"


# ============================================================
# 7. DOWNLOAD WAVEFORM (UNTUK SATU TASK)
# ============================================================

def download_task(task):
    """
    task: dict {
        'event': radar_event_dict,
        'hybrid_row': hybrid_row_dict,
        'station': sta_code
    }
    """
    ev = task["event"]
    hrow = task["hybrid_row"]
    sta_code = task["station"]

    if hrow["source"] != "BMKG+USGS":
        return {
            "Event_ID": ev["Event_ID"],
            "BMKG_Index": int(hrow["bmkg_index"]),
            "source": hrow["source"],
            "station": sta_code,
            "status": "SKIP_BMKG_ONLY"
        }

    if pd.isna(hrow["usgs_time"]):
        return {
            "Event_ID": ev["Event_ID"],
            "BMKG_Index": int(hrow["bmkg_index"]),
            "source": hrow["source"],
            "station": sta_code,
            "status": "SKIP_NO_USGS_TIME"
        }

    # Auto-detect station active
    health = check_station_active(sta_code)
    if health != "ACTIVE":
        return {
            "Event_ID": ev["Event_ID"],
            "BMKG_Index": int(hrow["bmkg_index"]),
            "source": hrow["source"],
            "station": sta_code,
            "status": "SKIP_STATION_INACTIVE"
        }

    event_time = UTCDateTime(hrow["usgs_time"].to_pydatetime())
    file_name = f"{sta_code}_{ev['Event_ID']}.mseed"
    file_path = os.path.join(SAVE_DIR, file_name)

    if os.path.exists(file_path):
        return {
            "Event_ID": ev["Event_ID"],
            "BMKG_Index": int(hrow["bmkg_index"]),
            "source": hrow["source"],
            "station": sta_code,
            "status": "EXIST"
        }

    t_start = event_time - 60
    t_end = event_time + 180

    for client in (client_iris, client_gfz):
        try:
            st = client.get_waveforms(
                network="*",
                station=sta_code,
                location="*",
                channel="BH?,HH?,EH?",
                starttime=t_start,
                endtime=t_end
            )
            if len(st) > 0:
                st.write(file_path, format="MSEED")
                return {
                    "Event_ID": ev["Event_ID"],
                    "BMKG_Index": int(hrow["bmkg_index"]),
                    "source": hrow["source"],
                    "station": sta_code,
                    "status": "OK"
                }
        except:
            continue

    return {
        "Event_ID": ev["Event_ID"],
        "BMKG_Index": int(hrow["bmkg_index"]),
        "source": hrow["source"],
        "station": sta_code,
        "status": "FAIL"
    }


# ============================================================
# 8. DASHBOARD HTML
# ============================================================

def build_dashboard(log_df: pd.DataFrame):
    if log_df.empty:
        html = "<html><body><h2>No data in log yet.</h2></body></html>"
        with open(DASHBOARD_HTML, "w") as f:
            f.write(html)
        return

    summary = log_df["status"].value_counts().to_frame("count")
    summary["percent"] = (summary["count"] / len(log_df) * 100).round(2)

    html = []
    html.append("<html><head><meta charset='utf-8'><title>Hybrid Download Dashboard</title>")
    html.append("<style>body{font-family:Arial, sans-serif;margin:20px;} table{border-collapse:collapse;} th,td{border:1px solid #ccc;padding:4px 8px;} th{background:#eee;} .ok{color:green;font-weight:bold;} .fail{color:red;font-weight:bold;} .skip{color:#999;} </style>")
    html.append("</head><body>")
    html.append("<h1>Hybrid Waveform Download Dashboard</h1>")

    html.append("<h2>Summary</h2>")
    html.append(summary.to_html(classes="summary", border=0))

    html.append("<h2>Sample Log (first 200 rows)</h2>")
    sample = log_df.head(200).copy()
    html.append(sample.to_html(classes="log", border=0, index=False))

    html.append("</body></html>")

    with open(DASHBOARD_HTML, "w") as f:
        f.write("\n".join(html))


# ============================================================
# 9. ORKESTRASI UTAMA (MULTI-THREADED + AUTO-SKIP)
# ============================================================

def run_hybrid_pipeline():
    bmkg, usgs = load_catalogs()
    hybrid = build_hybrid_catalog(bmkg, usgs)
    radar_events = load_radar_events()
    hybrid_lookup = build_hybrid_lookup(hybrid)

    tasks = []

    print("🧩 Menyusun task download (auto-skip event yang tidak match BMKG)...")
    for ev in tqdm(radar_events, desc="Build tasks", unit="event"):
        t_radar = pd.to_datetime(ev["Time_UTC"], utc=True)

        dt = (bmkg["origin_time"] - t_radar).abs()
        closest_idx = dt.idxmin()

        if dt.min() > RADAR_MATCH_TOL:
            # event radar tidak match BMKG → auto-skip
            continue

        bmkg_idx = int(closest_idx)
        if bmkg_idx not in hybrid_lookup:
            continue

        hrow = hybrid_lookup[bmkg_idx]

        # kalau BMKG_ONLY, kita masih boleh buat task, tapi nanti status SKIP_BMKG_ONLY
        for sta_info in ev["Stations"]:
            sta_code = sta_info["station"]
            tasks.append({
                "event": ev,
                "hybrid_row": hrow,
                "station": sta_code
            })

    print(f"   Total tasks: {len(tasks)}")

    logs = []
    print(f"🚀 Mulai unduhan multi-threaded (workers={MAX_WORKERS})...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(download_task, t) for t in tasks]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Downloading", unit="task"):
            res = fut.result()
            logs.append(res)

    log_df = pd.DataFrame(logs)
    log_df.to_csv(LOG_PATH, index=False)
    print(f"\n✅ Selesai. Log disimpan di: {LOG_PATH}")

    print("📊 Membangun dashboard HTML...")
    build_dashboard(log_df)
    print(f"✅ Dashboard: {DASHBOARD_HTML}")


# ============================================================
# 10. MAIN
# ============================================================

if __name__ == "__main__":
    run_hybrid_pipeline()


📥 Load katalog BMKG & USGS...
   BMKG events: 216716
   USGS events: 62358
🔗 Matching BMKG → USGS (build katalog hybrid)...


Matching: 100%|██████████| 216716/216716 [01:14<00:00, 2898.83ev/s]


   Hybrid total: 216716
   BMKG+USGS   : 23590
   BMKG_ONLY   : 193126
📡 Radar events: 13547
🚀 Mulai unduhan waveform (Hybrid Mode)...


Events:   0%|          | 38/13547 [05:07<30:19:35,  8.08s/event, GSI [OK]]             


KeyboardInterrupt: 